# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SinhaK\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\SinhaK\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [22]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

c:\Users\SinhaK\AppData\Local\miniconda3\envs\agentenv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
%pip install -U ragas langchain-openai langchain-community langchain-text-splitters langchain-qdrant qdrant-client langsmith openevals nltk tiktoken openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_12016\4068800016.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_12016\4068800016.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [12]:
!pip install rapidfuzz

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.1 MB/s  0:00:00


In [13]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\SinhaK\AppData\Local\miniconda3\envs\agentenv312\Lib\site-packages\langsmith\client.py:498: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019c6ee4-dbcf-70f1-8c32-53753dc1edd5,id=019c6ee4-dbcf-70f1-8c32-53753dc1edd5; trace=019c6ee4-dbcf-70f1-8c32-53753dc1edd5,id=019c6ee4-dcbc-7eb3-aa9f-e2356a74d843; trace=019c6ee4-dcc1-7340-b8ea-1e85ca49ba6d,id=019c6ee4-dcc1-7340-b8ea-1e85ca49ba6d; trace=019c6ee4-dcc1-7340-b8ea-1e85ca49ba6d,id=019c6ee4-dcc2-7ef0-9a56-48c3addd8355
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 74.76it/s]


KnowledgeGraph(nodes: 12, relationships: 34)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019c6ee5-0985-7110-8108-66b08e450cc3,id=019c6ee5-0986-7112-a126-095f14ba120b; trace=019c6ee5-0985-7110-8108-66b08e450cc3,id=019c6ee5-0985-7110-8108-66b08e450cc3; trace=019c6ee5-0984-70a2-ab9e-ffd14f842004,id=019c6ee5-0984-70a2-ab9e-ffefa97e8220; trace=019c6ee5-0987-7e43-922d-b997c9675e4d,id=019c6ee5-0987-7e43-922d-b9ad1c113cfe; trace=019c6ee5-0980-7901-a63d-80d3ba07e4b2,id=019c6ee5-0980-7901-a63d-80e970da90b4; trace=019c6ee5-0984-70a2-ab9e-ffd14f842004,id=019c6ee5-0984-70a2-ab9e-ffd14f842004; trace=019c6ee5-0987-7e43-922d-b997c9675e4d,id=019c6ee5-0987-7e43-922d-b997c9675e4d; trace=019c6ee5-0980-7901-a63d-80d3ba07e4b2,id=019c6ee5-0980-7901-a63d-80d3ba07e4b2; trace=019c6ee5-0988-7a03-9e7c-804d115d3a29,id

We can save and load our knowledge graphs as follows.

In [14]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 12, relationships: 34)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [16]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
## 1. SingleHopSpecificQuerySynthesizer



Makes straightforward, specific questions that can be answered from one chunk / one node in the knowledge graph (a “single hop”).

Think: “What is X?” “When did Y happen?”—a fact that lives in one place. 


## 2. MultiHopAbstractQuerySynthesizer



Makes multi-hop questions that require pulling info from multiple related chunks and doing higher-level reasoning (themes, comparisons, patterns).

Think: “How do A and B differ conceptually?” “What’s the overall trend across these sections?” 


## 3. MultiHopSpecificQuerySynthesizer



Makes multi-hop questions that still aim for a specific, grounded answer, but the answer requires connecting specific facts from multiple chunks (often via entity/keyphrase overlap).

Think: “Which X mentioned in doc A is impacted by Y described in doc B?”—precise, but needs stitching. 

Finally, we can use our `TestSetGenerator` to generate our testset!

In [17]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples: 100%|██████████| 11/11 [00:04<00:00,  2.26it/s]


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,How does cardio exercise contribute to mental ...,[The Personal Wellness Guide\nA Comprehensive ...,"Exercise, including cardio, is one of the most...",Mental Health Advocate,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,How do Proteins help in muscle repair and immu...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Proteins are essential for muscle repair and i...,Mental Health Advocate,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
2,Can you explain how fatigue is related to stre...,[and mental health. Physical symptoms of stres...,Fatigue is listed as a physical symptom of str...,Mental Health Advocate,MISSPELLED,LONG,single_hop_specific_query_synthesizer
3,What is a recommended beginner workout schedul...,[Chapter 3: Building a Workout Routine\n\nStar...,The beginner weekly schedule for Monday includ...,Mental Health Advocate,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,How does sunlight benefit mental health?,[A strong immune system helps protect against ...,"Sunlight provides vitamin D, which is importan...",Mental Health Advocate,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
5,How do the foundations of mental health relate...,[<1-hop>\n\nThe Mental Health and Psychology H...,The handbook explains that the foundations of ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How does maintaining proper hydration support ...,[<1-hop>\n\nThe Personal Wellness Guide\nA Com...,The personal wellness guide emphasizes that wa...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How growth mindset and habit formation help me...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,Growth mindset encourages positive thinking an...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,how does chapter 11 talk about stress reductio...,[<1-hop>\n\nand mental health. Physical sympto...,Chapter 11 discusses stress reduction techniqu...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,Considering the comprehensive connection betwe...,[<1-hop>\n\nChapter 3: The Mind-Body Connectio...,The chapters highlight that mental health is d...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [18]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Generating Samples: 100%|██████████| 12/12 [00:05<00:00,  2.31it/s]


In [19]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,so like what exercise and movement stuff shoul...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Exercise and movement are important for health...,Mental Health Awareness Coach,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What is CBT-I and how does it help improve sle...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Cognitive Behavioral Therapy for Insomnia (CBT...,Mental Health Advocate,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,How can I manage headaches naturally like drin...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Natural headache remedies include drinking wat...,Mental Health Awareness Coach,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,What is the chaper 12 about in the context of ...,[PART 4: STRESS MANAGEMENT AND MENTAL WELLNESS...,Chapter 12 discusses mindfulness and meditatio...,Mental Health Awareness Coach,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
4,How do components of emotional intelligence re...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,The context explains that emotional intelligen...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,What are the common mental health conditions a...,[<1-hop>\n\nThe Mental Health and Psychology H...,Common mental health conditions include anxiet...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,"How do the key nutrients for brain health, lik...",[<1-hop>\n\non mindset has significant implica...,"The key nutrients for brain health, such as om...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How do proper hydration and water intake suppo...,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,The nutrition chapter emphasizes that water is...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,Can you tell me how chapter 9 about understand...,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,Chapter 9 explains that insomnia involves diff...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,PART 5 how build good habits and morning routi...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,PART 5 explains that building healthy habits i...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
1. Unrolled (manual): you explicitly build the Knowledge Graph, explicitly run transforms, explicitly set your query synthesizers / distribution, then generate the testset.

2. Abstracted (automatic): you call a higher-level API and Ragas does most of the wiring (KG creation + transforms + default query distribution) for you.


Here are the trade-offs and when to use each:

### Unrolled (manual) approach — trade-offs

## Pros

1. Maximum control & customization: you can pick chunking strategy, which extractors/transforms run, how relationships are built, and how query types are distributed. 

2. Better debugging/observability: if quality is off (bad chunks, weak entity links, weird questions), you can inspect/save the KG and pinpoint which step caused it. 

3. Domain-specific tuning: you can swap in a custom splitter (e.g., section-aware for financial docs) or custom query synthesizers. 


### Cons

1. More code + more moving parts: higher setup time and more chance of misconfiguration.

2. You own the “defaults”: you have to decide what’s “good enough” for transforms and distributions.


### Choose manual when

1. You’re building a production-grade evaluation dataset and need repeatability, explainability, and tuning.

2. Your documents are domain-specific (finance/legal/medical) and need custom chunking/transforms. 

3. You need to debug why generated questions/answers look wrong.



---

### Abstracted (automatic) approach — trade-offs

### Pros

1. Fastest time-to-results: minimal code to get a working synthetic set.

2. Good defaults: Ragas provides a default query distribution (single-hop specific + multi-hop abstract + multi-hop specific) so you quickly cover common query patterns. 

3. Great for iteration: quick smoke-tests while you’re still developing the RAG pipeline.


### Cons

1. Less control: harder to tweak KG creation/transforms/chunking without dropping down to the unrolled pipeline. 

2. Harder to diagnose quality issues: when something is off, you may not see which internal step created the problem unless you switch to manual.


### Choose automatic when

1. You want a quick baseline evaluation set for a new RAG system.

2. You’re in early prototyping and care more about speed than perfect coverage.

3. You don’t yet know what customizations you need.

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [20]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default
# 🏗️ Activity #1: Custom Query Distribution (single-cell, copy/paste)

from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)

# ---------- 1) Define default + custom query distributions ----------
default_dist = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.50),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

# Custom: more multi-hop (harder, more realistic RAG eval)
custom_dist = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.30),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.30),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.40),
]

# ---------- 2) Generate test sets ----------
TESTSET_SIZE = 30 # increase to 50/100 if you want ratios to show up more clearly

testset_default = generator.generate(testset_size=TESTSET_SIZE, query_distribution=default_dist)
testset_custom = generator.generate(testset_size=TESTSET_SIZE, query_distribution=custom_dist)

df_default = testset_default.to_pandas()
df_custom = testset_custom.to_pandas()

print("✅ Generated test sets!")
print("Default rows:", len(df_default), "| Custom rows:", len(df_custom))

# ---------- 3) Compare question type distributions ----------
def summarize_types(df, name="dataset"):
    # Common column names across ragas versions
    candidate_cols = ["synthesizer", "query_type", "type", "evolution_type", "question_type"]
    col = next((c for c in candidate_cols if c in df.columns), None)

    print(f"\n=== {name} ===")
    if col:
        print(f"Type column detected: '{col}'")
        print(df[col].value_counts(dropna=False))
    else:
        # best-effort fallback: check for metadata-like columns
        meta_col = next((c for c in df.columns if "meta" in c.lower()), None)
        if meta_col:
            print(f"No explicit type column found. Metadata column detected: '{meta_col}'")
            print("Sample metadata:")
            print(df[meta_col].head(3))
        else:
            print("No explicit type/metadata column found in this ragas output.")
            print("We will do a qualitative comparison via sample questions.")

summarize_types(df_default, "Default distribution")
summarize_types(df_custom, "Custom distribution")

# ---------- 4) Qualitative comparison: sample questions ----------
qcol = "question" if "question" in df_default.columns else df_default.columns[0]

print("\n--- Sample questions (Default) ---")
for q in df_default[qcol].head(10):
    print("-", q)

print("\n--- Sample questions (Custom) ---")
for q in df_custom[qcol].head(10):
    print("-", q)

# ---------- 5) Explanation for chosen weights ----------
explanation = """
Why these weights?
- MultiHopSpecific (0.40): Emphasizes grounded multi-step questions that require retrieving + connecting facts across chunks.
- MultiHopAbstract (0.30): Adds higher-level synthesis/comparison/why-how questions—common real-world RAG use.
- SingleHopSpecific (0.30): Keeps basic factual lookups for coverage, but reduces "easy" questions vs the default.
"""
print("\n" + explanation.strip())


Generating Samples: 100%|██████████| 30/30 [00:10<00:00,  2.85it/s]

✅ Generated test sets!
Default rows: 32 | Custom rows: 30

=== Default distribution ===
No explicit type/metadata column found in this ragas output.
We will do a qualitative comparison via sample questions.

=== Custom distribution ===
No explicit type/metadata column found in this ragas output.
We will do a qualitative comparison via sample questions.

--- Sample questions (Default) ---
- What does Chapter 1 cover in the context of exercise and movement?
- What is PART  1 about exercise and movement?
- So, like, carbs are they like energy stuff or what, and why we need them for good health?
- What are some herbal teas recommended for improving sleep quality?
- What information is covered in Chapter 14 regarding morning routines for wellness, and how can understanding this chapter help individuals improve their emotional and psychological well-being?
- How can morning routines for wellness help me feel better?
- What are the key topics covered in Chapter 10 of the stress management and

1. I set MultiHopSpecific to 0.40 because it best simulates real RAG behavior: users often ask questions where the answer is specific, but you must pull multiple pieces of evidence from different chunks and connect them correctly.

2. I set MultiHopAbstract to 0.30 to ensure the testset also contains “synthesis” questions (summary/comparison/why/how). These are common failure modes for LLMs even when retrieval succeeds.

3. I kept SingleHopSpecific at 0.30 so we still test simple factual lookups and baseline retrieval correctness, but not so many that the dataset becomes “too easy.”

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [35]:
import os, uuid
from getpass import getpass
from langsmith import Client

# Force US endpoint (your key works here)
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

# Ensure key is set (strip whitespace)
if not os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("Paste LangSmith API key: ").strip()
else:
    os.environ["LANGSMITH_API_KEY"] = os.environ["LANGSMITH_API_KEY"].strip()

# Create client using explicit args (avoids any stale config)
client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"],
)

# Sanity check
_ = list(client.list_datasets(limit=1))
print("✅ LangSmith SDK auth confirmed.")

# Create dataset
dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"
langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases",
)
print("✅ Created dataset:", langsmith_dataset.name)
print("Dataset ID:", langsmith_dataset.id)

✅ LangSmith SDK auth confirmed.
✅ Created dataset: Use Case Synthetic Data - AIE9 - c2c15966-c71c-43d9-875c-0ece71b4f614
Dataset ID: cc7f74da-7c18-487a-af10-24310121dde4


We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [36]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [37]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [40]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [41]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [42]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [43]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [44]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [45]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [46]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [47]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [49]:
import os
from langsmith import Client
from langsmith.evaluation import evaluate
from openevals.llm import create_llm_as_judge

# --- Force LangSmith config (US) + create an authenticated client explicitly ---
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
assert os.getenv("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY is not set in this kernel. Set it before running evaluation."

ls_client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"].strip(),
)

# --- Resolve dataset by NAME -> ID using the same authenticated client ---
# IMPORTANT: dataset_name must exactly match what you created earlier
ds = ls_client.read_dataset(dataset_name=dataset_name)
print("✅ Using dataset:", ds.name)
print("Dataset ID:", ds.id)

# --- Evaluators (same as your code) ---
qa_evaluator = create_llm_as_judge(
    prompt=(
        "You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\n"
        "Input: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\n"
        "Is the prediction correct? Return 1 if correct, 0 if incorrect."
    ),
    feedback_key="qa",
    model="openai:gpt-4o",
)

labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o",
)

dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o",
)

# --- Run evaluation using the SAME authenticated client ---
results = evaluate(
    rag_chain.invoke,
    data=ds.id, # pass dataset id to avoid name lookup issues
    evaluators=[qa_evaluator, labeled_helpfulness_evaluator, dopeness_evaluator],
    metadata={"revision_id": "default_chain_init"},
    client=ls_client, # <-- THIS is the key fix
)

print("✅ Evaluation started / completed.")
results

✅ Using dataset: Use Case Synthetic Data - AIE9 - c2c15966-c71c-43d9-875c-0ece71b4f614
Dataset ID: cc7f74da-7c18-487a-af10-24310121dde4
View the evaluation results for experiment: 'diligent-committee-94' at:
https://smith.langchain.com/o/bea48619-aa1e-4493-9296-d1f4cb26e708/datasets/cc7f74da-7c18-487a-af10-24310121dde4/compare?selectedSessions=667956e4-4754-4bc1-90f0-d0997d74a13c




12it [02:23, 11.93s/it]

✅ Evaluation started / completed.


,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does understanding the factors that contri...,Understanding the factors that contribute to m...,None,The Mental Health and Psychology Handbook expl...,True,True,True,4.097620,88a94db0-23de-44e0-856c-68c2f3832a1c,019c6f1a-ffde-7571-b123-8d810bb08c1d
1,Based on Chapter 10's discussion of stress and...,"Based on the context, mindfulness and meditati...",None,Chapter 10 explains that mental symptoms of st...,True,True,True,3.746890,3f977d44-ff9e-4de8-a3ec-4ea3f6ebe43a,019c6f1b-2d08-7e62-aab6-ec798b6fdf90
2,PART 5 how build good habits and morning routi...,Based on the provided context:\n\nPART 5 cover...,None,PART 5 explains that building healthy habits i...,True,True,True,5.596815,dc26b741-ffb6-4d94-82ce-0f7ad829abeb,019c6f1b-578c-7a22-a0a7-46837685ca7f
3,Can you tell me how chapter 9 about understand...,Chapter 9 defines insomnia as difficulty falli...,None,Chapter 9 explains that insomnia involves diff...,True,True,False,4.762728,06d44bca-5ddb-4009-b9db-ef4fb0d86d81,019c6f1b-8add-71b2-855f-3416d5cb2707
4,How do proper hydration and water intake suppo...,Proper hydration supports overall health by be...,None,The nutrition chapter emphasizes that water is...,True,True,False,3.700915,2015cc50-c2fe-4e16-bb7f-fe026a216ab3,019c6f1b-bceb-7281-8f7e-9c5cf23a0be3
5,"How do the key nutrients for brain health, lik...","The key nutrients for brain health, such as om...",None,"The key nutrients for brain health, such as om...",True,True,True,4.432343,88ba1470-b560-4380-b93f-367db6ccc502,019c6f1b-fd40-7fd1-9766-9f92fc6e540e
6,What are the common mental health conditions a...,Common mental health conditions include anxiet...,None,Common mental health conditions include anxiet...,True,True,False,3.787770,fb723bd1-d165-4787-b04a-1132ffc5b2b5,019c6f1c-340a-7ab1-8d3a-23dab48df5f3
7,How do components of emotional intelligence re...,The context does not explicitly explain how th...,None,The context explains that emotional intelligen...,False,False,False,1.801633,862f1387-53a8-4acc-bf82-686a181e794f,019c6f1c-63a0-7f80-aebe-a273a093556f
8,What is the chaper 12 about in the context of ...,Chapter 12 in the context of stress management...,None,Chapter 12 discusses mindfulness and meditatio...,True,True,False,1.236896,766f0d31-45aa-46e7-b0cc-d86c061f71f9,019c6f1c-81e6-7a70-8c50-0a1b9a905037
9,How can I manage headaches naturally like drin...,You can manage headaches naturally by:\n\n- Dr...,None,Natural headache remedies include drinking wat...,True,True,False,1.749880,2596a878-a88b-40db-b66a-95d29624e59d,019c6f1c-aeb1-77a2-a504-c3f2ce7cd4fb


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [51]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [52]:
rag_documents = docs

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:

Because chunk size changes what gets retrieved and what the LLM has to read, which directly impacts accuracy, latency, and cost.

1. Retrieval precision vs recall

Smaller chunks → more precise matches (less noise), but you might miss needed context if the answer spans multiple chunks. Larger chunks → more context included (better recall), but the chunk may contain lots of irrelevant text, making retrieval noisier.

---

2. Answer completeness

Some questions need a full paragraph/section to answer well. Too-small chunks can lead to partial answers.Too-large chunks can bury the key sentence inside unrelated content, so the model may miss it or get distracted.

---

3. Embedding + similarity behavior

Embeddings represent the whole chunk as one vector. If the chunk is large and covers multiple topics, the vector becomes “averaged,” and similarity search gets less sharp.

---

4. Context window / token budget

Bigger chunks mean fewer chunks can fit into the prompt. You may retrieve fewer distinct sources, losing coverage.

Smaller chunks let you fit more evidence, but can increase prompt assembly and reranking overhead.

---

5. Latency & cost

Large chunks → more tokens to send to the LLM → higher cost and slower responses.

Very small chunks → more retrieval calls / reranking / stitching → can also increase latency.


In [54]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Because the embedding model defines how your text is represented in vector space, which determines whether the right context is retrieved for a query.

1. Retrieval quality depends on semantic representation

Better embedding models capture meaning, paraphrases, domain terms, and relationships more accurately.

A weaker model may retrieve “keyword-ish” matches instead of the truly relevant passage.


2. Domain fit matters

Some embeddings handle technical/medical/legal language better than others.

If your docs are domain-heavy (internal acronyms, product names), a model that understands those patterns will retrieve better.


3. Chunk similarity ranking changes

Different models produce different vector geometry.

That changes which chunks are “closest,” affecting top-k retrieval, which then affects answer accuracy.


4. Multilingual / style robustness

Some embeddings are better with different writing styles, abbreviations, or mixed-language text. That impacts recall.


5. Speed + cost trade-offs

Smaller/faster embedding models reduce latency and cost but may lower retrieval accuracy.

Larger models can improve retrieval but increase compute time/cost.


Index consistency

If you change embedding models, you must re-embed and rebuild the vector index. Mixing old vectors with new ones breaks similarity search.



In short: chunk size controls the granularity of “what can be retrieved,” and embedding models control the intelligence of “how retrieval matches meaning.” Both directly affect retrieval quality, which is the biggest driver of end-to-end RAG performance.

In [55]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [56]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [57]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [58]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Yo, let’s crank your sleep game to legendary status! Here’s the ultimate cheat code straight from the wisdom vault of sleep masters:\n\n1. **Lock down your sleep schedule** like it’s a sacred ritual—hit the sack and rise at the same time every day, even when Netflix begs you to binge on weekends.\n\n2. **Craft a bedtime ritual that’s pure zen:** think cozy reading, gentle stretching, or a warm bath that melts stress like butter.\n\n3. **Turn your bedroom into a sleep fortress:** keep it cool (65-68°F / 18-20°C), pitch black (blackout curtains or a sleep mask are your allies), and ultra-quiet (white noise machines or earplugs can be game-changers).\n\n4. **Ghost the screens before bed:** ditch blue light at least 1-2 hours before lights out so your melatonin can throw the ultimate sleep party.\n\n5. **Cut the caffeine cutoff at 2 PM:** no late-afternoon energy boosts sabotaging your snooze vibes.\n\n6. **Work that body earlier in the day:** regular exercise improves sleep, but don’t hi

Finally, we can evaluate the new chain on the same test set!

In [60]:
import os
from operator import itemgetter
from langchain_qdrant import QdrantVectorStore
from langsmith import Client
from langsmith.evaluation import evaluate

# -----------------------
# 1) Build Qdrant vectorstore + retriever (in-memory)
# -----------------------
vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs",
)

retriever = vectorstore.as_retriever()

dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt
    | llm
    | StrOutputParser()
)

print("Chain sanity check:")
print(dopeness_rag_chain.invoke({"question": "How can I improve my sleep quality?"})[:300])

# -----------------------
# 2) Create an authenticated LangSmith client (force US endpoint)
# -----------------------
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
assert os.getenv("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY missing in this kernel"

ls_client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"].strip(),
)

# Resolve dataset using the SAME client to avoid internal token issues
ds = ls_client.read_dataset(dataset_name=dataset_name)
print("\n✅ Using dataset:", ds.name)
print("Dataset ID:", ds.id)

# -----------------------
# 3) Evaluate using the SAME authenticated client
# -----------------------
results = evaluate(
    dopeness_rag_chain.invoke,
    data=ds.id, # pass dataset id (best)
    evaluators=[qa_evaluator, labeled_helpfulness_evaluator, dopeness_evaluator],
    metadata={"revision_id": "dopeness_rag_chain"},
    client=ls_client, # <-- critical fix
)

print("\n✅ Evaluation started/completed.")
results

Chain sanity check:
Yo, wanna level up your sleep game and crash like a champ? Here’s the secret sauce straight outta the sleep playbook:

1. **Lock in a consistent sleep schedule** — your body’s a rhythm machine, so hit the sack and rise *same time* every day, weekends included. No excuses.

2. **Craft the ultimate be

✅ Using dataset: Use Case Synthetic Data - AIE9 - c2c15966-c71c-43d9-875c-0ece71b4f614
Dataset ID: cc7f74da-7c18-487a-af10-24310121dde4
View the evaluation results for experiment: 'notable-space-1' at:
https://smith.langchain.com/o/bea48619-aa1e-4493-9296-d1f4cb26e708/datasets/cc7f74da-7c18-487a-af10-24310121dde4/compare?selectedSessions=abb337db-8032-4c6b-b85b-be88df73b212




12it [02:51, 14.27s/it]


✅ Evaluation started/completed.


,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does understanding the factors that contri...,"Alright, let’s break it down in a way that’s s...",None,The Mental Health and Psychology Handbook expl...,True,True,True,5.986925,88a94db0-23de-44e0-856c-68c2f3832a1c,019c6f2a-9bf8-70b0-8af4-baa423ab3bc8
1,Based on Chapter 10's discussion of stress and...,"Yo, here’s the dopest breakdown for flexing mi...",None,Chapter 10 explains that mental symptoms of st...,True,True,True,6.437635,3f977d44-ff9e-4de8-a3ec-4ea3f6ebe43a,019c6f2a-d434-7a40-85fd-9fc3a5bff438
2,PART 5 how build good habits and morning routi...,"Alright, buckle up—here’s the ultimate lowdown...",None,PART 5 explains that building healthy habits i...,True,True,True,7.399615,dc26b741-ffb6-4d94-82ce-0f7ad829abeb,019c6f2b-148a-74b1-a2f9-235abeaf4088
3,Can you tell me how chapter 9 about understand...,"Yo, strap in—here’s the ultimate sleep science...",None,Chapter 9 explains that insomnia involves diff...,True,True,True,6.604929,06d44bca-5ddb-4009-b9db-ef4fb0d86d81,019c6f2b-5888-74b0-897f-036738ed46be
4,How do proper hydration and water intake suppo...,"Alright, buckle up, because hydration isn’t ju...",None,The nutrition chapter emphasizes that water is...,True,True,True,5.044855,2015cc50-c2fe-4e16-bb7f-fe026a216ab3,019c6f2b-97d1-79a0-9446-2b22efe5d659
5,"How do the key nutrients for brain health, lik...","Alright, let’s dive deep and rock this mental ...",None,"The key nutrients for brain health, such as om...",True,True,True,6.536147,88ba1470-b560-4380-b93f-367db6ccc502,019c6f2b-c6b9-7a50-b502-445b6de818c0
6,What are the common mental health conditions a...,"Yo, let’s dive into the mental health vibe wit...",None,Common mental health conditions include anxiet...,True,True,True,5.692992,fb723bd1-d165-4787-b04a-1132ffc5b2b5,019c6f2c-0142-7833-a914-66ac2d55da28
7,How do components of emotional intelligence re...,"Alright, let’s crank this up to eleven and bre...",None,The context explains that emotional intelligen...,True,True,True,6.448735,862f1387-53a8-4acc-bf82-686a181e794f,019c6f2c-310f-7382-82e1-6c11a4187715
8,What is the chaper 12 about in the context of ...,"Chapter 12 is straight-up the zen zone, diving...",None,Chapter 12 discusses mindfulness and meditatio...,True,True,True,1.612960,766f0d31-45aa-46e7-b0cc-d86c061f71f9,019c6f2c-691b-73f1-85b4-67182a44d70f
9,How can I manage headaches naturally like drin...,"Alright, here’s the ultimate headache hack fro...",None,Natural headache remedies include drinking wat...,True,True,True,2.901610,2596a878-a88b-40db-b66a-95d29624e59d,019c6f2c-9195-7fe1-a060-6d0a42f1d2a9


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
Screenshot A (baseline chain): diligent-committee-94

Shows dopeness avg ≈ 0.417, while helpfulness avg ≈ 0.917 and qa avg ≈ 0.917.

Several rows have dopeness = 0.00 (highlighted red), even though helpfulness/qa are often 1.00.


Screenshot B (dopeness chain): notable-space-1

Shows dopeness avg = 1.00, helpfulness avg = 1.00, and qa avg = 1.00 across the visible rows.

Outputs visibly use a more “dope” style (“Yo…”, “Alright…”, slang / hype tone), matching the evaluator rubric.


(These two screenshots are the required evidence: they clearly show different metric outcomes and response styles.)


---

Why the metrics changed

1) Dopeness increased from ~0.417 → 1.00

This change is expected because the second chain is explicitly optimized for the dopeness criterion.

In the baseline chain, the assistant responses are mostly “normal/helpful” and often neutral/clinical. That can still be correct and helpful, but it fails the dopeness judge prompt (“dope, lit, cool vs generic”), so many rows get 0.00.

In the dopeness chain, the output tone is intentionally styled with slang/hype phrasing (“Yo, strap in…”, “Alright, buckle up…”). This aligns directly with the evaluator definition, so the judge consistently returns 1.00.


Interpretation: Dopeness didn’t improve because retrieval got smarter; it improved because you changed the response style to match what the metric rewards.


---

2) Helpfulness improved from ~0.917 → 1.00

Two likely reasons:

Tone can be perceived as more helpful. Even if content is similar, a more engaging, confident, structured voice often scores higher on “helpfulness” in LLM-as-judge settings.

The dopeness chain responses look more “complete” and “coach-like” (based on the visible snippets), which tends to satisfy “helpfulness” rubrics better.


Important nuance: This doesn’t necessarily mean the chain is objectively more helpful in the real world—LLM judges often treat clarity + confidence + coverage as “helpful,” even if the underlying retrieval is unchanged.


---

3) QA stayed strong (~0.917 → 1.00)

Your screenshots show QA was already high in the baseline chain and becomes perfect in the dopeness chain (at least for the visible rows).

Why it likely improved slightly:

The dopeness chain might still be using the same core factual content, but phrasing could be closer to the reference answer format, making the judge more likely to score it as correct.

Also, with small sample sizes, moving from 0.917 to 1.00 can reflect one or two borderline items flipping.


Key point: QA didn’t jump massively because the chain didn’t fundamentally change retrieval quality; it mainly changed presentation style.


---

4) Latency increased (about ~3.77s p50 → ~6.21s p50)

The dopeness chain shows higher median latency.

Most likely causes:

The dopeness chain is doing extra work in the pipeline (RAG retrieval step + generation), or it’s producing longer outputs (more tokens), which increases generation time.

If your dopeness chain adds a retriever call (Qdrant) where the baseline chain was simpler, that adds overhead even in-memory.


Interpretation: You traded speed for higher “style” scores and slightly better judge metrics.


---

Overall conclusion

The two chains differ mainly in response style alignment with the evaluator prompts. The dopeness-optimized chain produces outputs that explicitly match the “dope/lit/cool” rubric, which drives dopeness to 1.00, and also nudges helpfulness/qa upward—but at the cost of higher latency.